# Classification Project: Online Shoppers Purchasing Intention
## Notebook 2: Nested Cross-Validation and Data-Leakage-Proof Pipelines

In this phase, we build the core machine learning architecture. Following the strict methodological rule **"Never do preprocessing before splitting D"**, we will encapsulate all data transformations—including Outlier Removal, Feature Scaling, and SMOTE—inside an `imblearn` Pipeline.

### Objectives:
1. **Hold-out Split:** Isolate 20% of the data as a completely untouched Test Set for the final evaluation (Notebook 3).
2. **Custom LOF Sampler:** Engineer a custom pipeline component to remove multivariate outliers dynamically only during the training phase of the Cross-Validation.
3. **Nested Cross-Validation:** Perform an unbiased comparison between three models as required by the project proposal: **Random Forest** (baseline), **SVM** (required), and **XGBoost** (advanced gradient boosting).
4. **Final Model Selection:** Train the winning architecture on the full training set and save it for Explainable AI (XAI) analysis.

> Why is this not Data Leakage? Encoding strings to numbers does not involve calculating any dataset-wide statistics (like means or variances). It is purely a format change, so it is safe to do before splitting the data.

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split

print("--- 1. DATA LOADING & ENCODING ---")
current_dir = os.path.dirname(os.path.abspath('__file__'))
file_path = os.path.join(current_dir, '../online_shoppers_intention.csv')

df = pd.read_csv(file_path).dropna().reset_index(drop=True)

# Basic Encoding (No statistical leakage here, just format conversion)
df['Weekend'] = df['Weekend'].astype(int)
df['Revenue'] = df['Revenue'].astype(int)
# One-Hot Encoding for the few string categorical variables
df = pd.get_dummies(df, columns=['Month', 'VisitorType'], drop_first=True)

# Drop 'Region' — chi-square test in Notebook 1 found p=0.32 (not statistically
# associated with Revenue), so it is excluded from the feature set.
X = df.drop(columns=['Revenue', 'Region'])
y = df['Revenue']

print("--- 2. STRICT HOLD-OUT SPLIT (D_train vs D_test) ---")
# We isolate 20% of the data. This set will NOT be seen by the Cross-Validation.
# Stratify ensures the 85/15 class imbalance is maintained in both splits.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

print(f"Training Set (D_train): {X_train.shape[0]} samples")
print(f"Test Set (D_test): {X_test.shape[0]} samples (LOCKED AWAY)")

print("--- 3. SAVING TEST SET TO DEDICATED DIRECTORY ---")
# Define the directory name
output_dir = os.path.join(current_dir, '../2_locked_test_data')

# Create the directory if it does not exist (prevents errors on multiple runs)
os.makedirs(output_dir, exist_ok=True)

# Define the full file paths
x_test_path = os.path.join(output_dir, 'X_test_locked.csv')
y_test_path = os.path.join(output_dir, 'y_test_locked.csv')

# Save the test set to the new directory for Notebook 3
X_test.to_csv(x_test_path, index=False)
y_test.to_csv(y_test_path, index=False)

print(f"Test data successfully saved inside: {output_dir}")

--- 1. DATA LOADING & ENCODING ---
--- 2. STRICT HOLD-OUT SPLIT (D_train vs D_test) ---
Training Set (D_train): 9864 samples
Test Set (D_test): 2466 samples (LOCKED AWAY)
--- 3. SAVING TEST SET TO DEDICATED DIRECTORY ---
Test data successfully saved inside: C:\Users\lbart\Desktop\Data-Mining-ML-project\jupyter\../2_locked_test_data


### 2. Engineering the Custom LOF Sampler
Scikit-Learn's `LocalOutlierFactor` is not natively designed to work inside a classification pipeline (it lacks a standard `transform` method). To respect the *"Zero Leakage"* rule, we must prevent the model from identifying outliers using test-fold data.

We solve this by creating a custom class that inherits from `BaseEstimator` and `SamplerMixin`. This tricks the `imblearn` pipeline into treating LOF as an under-sampling technique.
* **During `fit` (Training Fold):** It calculates densities, flags outliers, and removes them from the training data.
* **During `predict` (Validation Fold):** The pipeline automatically skips the sampler, forcing the model to predict on real-world, uncleaned data (including natural outliers).


In [2]:
# Load the shared LOF_Sampler utility (single source of truth for all notebooks).
%run utils/lof_sampler.py

LOF_Sampler loaded from utils/lof_sampler.py


### 3. Nested Cross-Validation & Pipeline Construction
We now construct the full rigorous pipeline:
`LOF_Sampler` -> `StandardScaler` -> `SMOTE` -> `Classifier`

We will compare **three models** using a **Nested Cross-Validation** approach, as required by the project proposal:
* **Random Forest** — baseline ensemble model.
* **SVM (RBF kernel)** — required by the project proposal, paired with SMOTE.
* **XGBoost** — advanced gradient boosting algorithm.

**Inner Loop (GridSearchCV):** Finds the optimal hyperparameters (including the SMOTE ratio and model-specific parameters).  
**Outer Loop (cross_val_score):** Evaluates the unbiased generalization performance of each tuned pipeline.

We optimize for the **Macro F1-Score** to ensure the model balances precision and recall on the minority class.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import make_scorer, f1_score
import warnings
warnings.filterwarnings('ignore') # Keep the output clean

print("--- PIPELINE INITIALIZATION ---")

# 1. Random Forest Pipeline
rf_pipe = ImbPipeline([
    ('lof', LOF_Sampler(n_neighbors=100, contamination=0.05)),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# 2. SVM Pipeline (required by the project proposal)
svm_pipe = ImbPipeline([
    ('lof', LOF_Sampler(n_neighbors=100, contamination=0.05)),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', SVC(random_state=42, probability=True))
])

# 3. XGBoost Pipeline
xgb_pipe = ImbPipeline([
    ('lof', LOF_Sampler(n_neighbors=100, contamination=0.05)),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss'))
])

# --- PARAMETER GRIDS (Inner Loop) ---
rf_grid = {
    'smote__sampling_strategy': [0.7, 1.0],
    'classifier__n_estimators': [100],
    'classifier__max_depth': [5, 10]
}

# SVM grid: C controls the regularisation margin,
# kernel controls the decision boundary shape.
svm_grid = {
    'smote__sampling_strategy': [0.7, 1.0],
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['rbf', 'linear']
}

xgb_grid = {
    'smote__sampling_strategy': [0.7, 1.0],
    'classifier__n_estimators': [100],
    'classifier__max_depth': [3, 5],
    'classifier__learning_rate': [0.05, 0.1]
}

# --- NESTED CROSS-VALIDATION ---
print("\nExecuting Nested Cross-Validation (Outer CV: 5 folds, Inner CV: 3 folds)...")
print("Note: SVM evaluation may take several minutes due to kernel computation.\n")
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scorer = make_scorer(f1_score, average='macro')

# Grid Searches (one per model)
rf_gs  = GridSearchCV(rf_pipe,  rf_grid,  cv=cv_inner, scoring=scorer, n_jobs=-1)
svm_gs = GridSearchCV(svm_pipe, svm_grid, cv=cv_inner, scoring=scorer, n_jobs=-1)
xgb_gs = GridSearchCV(xgb_pipe, xgb_grid, cv=cv_inner, scoring=scorer, n_jobs=-1)

# Outer Loop Evaluation
rf_scores  = cross_val_score(rf_gs,  X_train, y_train, cv=cv_outer, scoring=scorer, n_jobs=-1)
print(f"Random Forest -> Macro F1-Score: {rf_scores.mean():.4f} (+/- {rf_scores.std() * 2:.4f})")

svm_scores = cross_val_score(svm_gs, X_train, y_train, cv=cv_outer, scoring=scorer, n_jobs=-1)
print(f"SVM (RBF)     -> Macro F1-Score: {svm_scores.mean():.4f} (+/- {svm_scores.std() * 2:.4f})")

xgb_scores = cross_val_score(xgb_gs, X_train, y_train, cv=cv_outer, scoring=scorer, n_jobs=-1)
print(f"XGBoost       -> Macro F1-Score: {xgb_scores.mean():.4f} (+/- {xgb_scores.std() * 2:.4f})")

# Determine Winner
model_scores = {
    'Random Forest': rf_scores.mean(),
    'SVM':           svm_scores.mean(),
    'XGBoost':       xgb_scores.mean()
}
best_model_name = max(model_scores, key=model_scores.get)
print(f"\nWINNER: {best_model_name} with Macro F1 = {model_scores[best_model_name]:.4f}!")

--- PIPELINE INITIALIZATION ---

Executing Nested Cross-Validation (Outer CV: 5 folds, Inner CV: 3 folds)...
Note: SVM evaluation may take several minutes due to kernel computation.



Random Forest -> Macro F1-Score: 0.8019 (+/- 0.0167)


SVM (RBF)     -> Macro F1-Score: 0.7986 (+/- 0.0222)


XGBoost       -> Macro F1-Score: 0.8118 (+/- 0.0172)

WINNER: XGBoost with Macro F1 = 0.8118!


### 4. Training the Final Production Model
Nested CV provides an unbiased *estimate* of our methodology's performance, but it does not return a single deployable model (it trains multiple models across the outer folds). 

Now that we have proven our methodology and selected the winning architecture, we will perform one final standard `GridSearchCV` over the **entire** $D_{train}$ set to find the absolute best hyperparameters. We will then save this final fitted pipeline to disk.

In [4]:
import json

print(f"--- TRAINING FINAL {best_model_name.upper()} MODEL ON ENTIRE D_TRAIN ---")

# Select the winning pipeline and grid based on nested CV results
if best_model_name == 'XGBoost':
    final_gs = GridSearchCV(xgb_pipe, xgb_grid, cv=cv_outer, scoring=scorer, n_jobs=-1)
elif best_model_name == 'SVM':
    final_gs = GridSearchCV(svm_pipe, svm_grid, cv=cv_outer, scoring=scorer, n_jobs=-1)
else:
    final_gs = GridSearchCV(rf_pipe, rf_grid, cv=cv_outer, scoring=scorer, n_jobs=-1)

# Fit on the entire training dataset
final_gs.fit(X_train, y_train)

print(f"Best Hyperparameters found:")
for param, value in final_gs.best_params_.items():
    print(f" - {param}: {value}")

print(f"Best Validation Macro F1-Score: {final_gs.best_score_:.4f}")

# Extract the absolute best pipeline
best_pipeline = final_gs.best_estimator_

# --- DYNAMIC PATHING FOR MODEL EXPORT ---
# Navigate one level up to the project root, then into the 'models' directory
models_dir = os.path.join(current_dir, '../models')
os.makedirs(models_dir, exist_ok=True) 

# Save the model (.pkl)
model_filename = os.path.join(models_dir, 'final_best_pipeline.pkl')
joblib.dump(best_pipeline, model_filename)

# Save the metadata (.json)
metadata = {
    "model_name": best_model_name,
    "validation_macro_f1": round(final_gs.best_score_, 4),
    "best_params": {str(k): str(v) for k, v in final_gs.best_params_.items()},
    "nested_cv_scores": {
        "Random Forest": round(model_scores['Random Forest'], 4),
        "SVM":           round(model_scores['SVM'], 4),
        "XGBoost":       round(model_scores['XGBoost'], 4)
    }
}
metadata_filename = os.path.join(models_dir, 'model_metadata.json')
with open(metadata_filename, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nFinal model successfully saved at: '{model_filename}'")
print(f"Metadata (including scores) successfully saved at: '{metadata_filename}'")
print("We are now ready to move to Notebook 3 for final Testing, Ablation, and SHAP Explainability!")

--- TRAINING FINAL XGBOOST MODEL ON ENTIRE D_TRAIN ---


Best Hyperparameters found:
 - classifier__learning_rate: 0.1
 - classifier__max_depth: 3
 - classifier__n_estimators: 100
 - smote__sampling_strategy: 0.7
Best Validation Macro F1-Score: 0.8147

Final model successfully saved at: 'C:\Users\lbart\Desktop\Data-Mining-ML-project\jupyter\../models\final_best_pipeline.pkl'
Metadata (including scores) successfully saved at: 'C:\Users\lbart\Desktop\Data-Mining-ML-project\jupyter\../models\model_metadata.json'
We are now ready to move to Notebook 3 for final Testing, Ablation, and SHAP Explainability!


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

os.makedirs('../figures', exist_ok=True)

# --- Nested CV Macro F1 comparison bar chart ---
model_names  = ['Random Forest', 'SVM (RBF/Linear)', 'XGBoost']

# Dynamically compute means from the fold scores calculated above
cv_f1_scores = [
    rf_scores.mean(),
    svm_scores.mean(),
    xgb_scores.mean()
]

# Compute the confidence interval (+-2 std dev) for each model
cv_errors = [
    rf_scores.std() * 2,
    svm_scores.std() * 2,
    xgb_scores.std() * 2
]

colors = ['#4878CF', '#6ACC65', '#D65F5F']
champion_idx = cv_f1_scores.index(max(cv_f1_scores))

fig, ax = plt.subplots(figsize=(8, 5))

# Add error bars via yerr and horizontal end-caps via capsize
bars = ax.bar(model_names, cv_f1_scores, color=colors, edgecolor='white', linewidth=0.8,
              yerr=cv_errors, capsize=7, error_kw={'elinewidth': 1.5, 'alpha': 0.8})

# Annotate each bar with its score label
for i, (bar, score, error) in enumerate(zip(bars, cv_f1_scores, cv_errors)):
    label = f'{score:.4f}'
    if i == champion_idx:
        label += ' ★'
    
    # Offset the label above the error bar top to avoid overlap
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + error + 0.003, 
            label, ha='center', va='bottom',
            fontsize=11, fontweight='bold' if i == champion_idx else 'normal')

ax.set_ylabel('Nested CV Macro F1-Score', fontsize=12)
ax.set_title('Nested Cross-Validation Results — Model Comparison', fontsize=14, fontweight='bold')

# Expand Y-axis to accommodate error bars and annotations
# Y-axis extended to 0.55 so the Sakar et al. benchmark line at 0.61 is fully visible.
# The visual gap between 0.61 and the bars (~0.80) also communicates the +31% improvement.
ax.set_ylim(0.55, 0.87)

ax.axhline(0.61, color='grey', linestyle='--', linewidth=1.2,
           label='Sakar et al. (2018) benchmark (0.61)')

ax.legend(fontsize=10, loc='upper left')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../figures/fig_04_cv_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to ../figures/fig_04_cv_scores.png")